# EDA & Data Engineering — Heat Pump Load Forecasting (Task 3)

## 0. Setup & Imports

In [ ]:
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13

DATA_PATH    = "data/data.csv"
DEVICES_PATH = "data/devices.csv"
SAMPLE_FRAC  = 0.10
RANDOM_SEED  = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Imports OK")

## 1. Data Loading (Memory-Safe)

In [ ]:
devices = pd.read_csv(DEVICES_PATH)
print(f"devices.csv — {devices.shape[0]} rows, {devices.shape[1]} cols")
devices.head()

In [ ]:
# Stratified sample: read in 500K-row chunks, keep 10% of each
CHUNK = 500_000
chunks = []

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK, low_memory=False):
    sampled = chunk.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED)
    chunks.append(sampled)

df = pd.concat(chunks, ignore_index=True)
del chunks

# Parse timedate (format: '2024-10-01 00:00:00 UTC')
df["timedate"] = pd.to_datetime(df["timedate"].str.replace(" UTC", "", regex=False), utc=True)

# Categorical dtypes
df["deviceType"] = df["deviceType"].astype("category")
df["x3"]         = df["x3"].astype("category")

# Merge device metadata
df = df.merge(devices[["deviceId", "latitude", "longitude"]], on="deviceId", how="left")

print(f"Sampled rows: {df.shape[0]:,}  | Columns: {df.shape[1]}")
print(f"Period distribution:\n{df['period'].value_counts().to_string()}")

In [ ]:
print(df.dtypes)
df.head(3)

## 2. Basic EDA

In [ ]:
# Missing values
missing_pct = df.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_pct):
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_pct.plot.bar(ax=ax, color="steelblue")
    ax.set_title("Missing values (%)")
    ax.set_ylabel("%")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

# Basic stats table
num_cols = df.select_dtypes(include="number").columns.tolist()
df[num_cols].describe().T.round(4)

In [ ]:
print(f"Unique devices: {df['deviceId'].nunique()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["deviceType"].value_counts().plot.bar(ax=axes[0], color="teal")
axes[0].set_title("Device count by type")
axes[0].set_xlabel("deviceType")

df["period"].value_counts().reindex(["train", "valid", "test"]).plot.bar(ax=axes[1], color="coral")
axes[1].set_title("Sample rows by period")
plt.tight_layout()
plt.show()

## 3. Target Variable (x2) Analysis

In [ ]:
train = df[df["period"] == "train"].copy()
print(f"Train rows: {len(train):,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train["x2"].hist(bins=80, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("x2 distribution (train)")
axes[0].set_xlabel("x2")

train["x2"].plot.kde(ax=axes[1], color="steelblue")
axes[1].set_title("x2 KDE (train)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=train, x="deviceType", y="x2", ax=ax, showfliers=False)
ax.set_title("x2 by deviceType (train)")
ax.set_xlabel("deviceType")
plt.tight_layout()
plt.show()

In [ ]:
train["month"] = train["timedate"].dt.month
train["hour"]  = train["timedate"].dt.hour
train["month_label"] = train["timedate"].dt.to_period("M").astype(str)
month_order = sorted(train["month_label"].unique())

fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=train, x="month_label", y="x2", order=month_order, ax=ax, showfliers=False)
ax.set_title("x2 by month (train)")
ax.set_xlabel("Month")
plt.tight_layout()
plt.show()

In [ ]:
hourly_avg = train.groupby("hour")["x2"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
hourly_avg.plot(ax=ax, marker="o", color="darkorange")
ax.set_title("Average x2 by hour of day (train)")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Mean x2")
plt.tight_layout()
plt.show()

In [ ]:
dev_means = train.groupby("deviceId")["x2"].mean().sort_values()
print("Bottom-10 devices (mean x2):")
print(dev_means.head(10).to_string())
print("\nTop-10 devices (mean x2):")
print(dev_means.tail(10).to_string())

## 4. Temporal Analysis

In [ ]:
monthly = train.groupby("month_label")["x2"].mean().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, marker="s", color="royalblue")
ax.set_title("Monthly mean x2 (train period)")
ax.set_xlabel("Month")
ax.set_ylabel("Mean x2")
plt.tight_layout()
plt.show()

In [ ]:
train["day_of_week"] = train["timedate"].dt.dayofweek
pivot = train.pivot_table(values="x2", index="day_of_week", columns="hour", aggfunc="mean")

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", linewidths=0.3,
            yticklabels=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_title("Mean x2 heatmap: day-of-week × hour (train)")
plt.tight_layout()
plt.show()

In [ ]:
daily = train.set_index("timedate").resample("D")[["x2", "t1"]].mean()

fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()
ax1.plot(daily.index, daily["x2"], color="royalblue",  label="x2 (load)")
ax2.plot(daily.index, daily["t1"], color="tomato",     label="t1 (outdoor temp)", alpha=0.7)
ax1.set_ylabel("Mean x2", color="royalblue")
ax2.set_ylabel("Mean t1 (normalised)", color="tomato")
ax1.set_title("Daily mean x2 vs outdoor temperature (t1) — train period")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

In [ ]:
# ACF for a sample device
try:
    from statsmodels.graphics.tsaplots import plot_acf
    sample_dev = train["deviceId"].value_counts().index[0]
    dev_series = (
        train[train["deviceId"] == sample_dev]
        .sort_values("timedate")["x2"]
        .dropna()
        .head(2000)
    )
    fig, ax = plt.subplots(figsize=(12, 3))
    plot_acf(dev_series, lags=50, ax=ax, title=f"x2 ACF — device {sample_dev[:8]}...")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("statsmodels not installed — skipping ACF plot")

## 5. Temperature Feature Analysis

In [ ]:
t_cols = [c for c in df.columns if c.startswith("t")] + ["x1", "x2"]
t_cols = [c for c in t_cols if c in df.columns and df[c].dtype not in ["object", "category"]]

fig, ax = plt.subplots(figsize=(14, 10))
corr = train[t_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, cmap="coolwarm", center=0,
            annot=True, fmt=".2f", linewidths=0.5, square=True)
ax.set_title("Feature correlation matrix (train)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col, ax in zip(["t1", "t2"], axes):
    for ml in month_order:
        subset = train[train["month_label"] == ml][col].dropna()
        if len(subset) > 10:
            subset.plot.kde(ax=ax, label=ml, alpha=0.7)
    ax.set_title(f"{col} distribution by month")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
sample_scatter = train.sample(min(5000, len(train)), random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sc = axes[0].scatter(sample_scatter["t1"], sample_scatter["x2"],
                     c=sample_scatter["month"], cmap="tab10", alpha=0.4, s=5)
axes[0].set_title("t1 vs x2 (coloured by month)")
axes[0].set_xlabel("t1 (outdoor temp)")
axes[0].set_ylabel("x2 (load)")
plt.colorbar(sc, ax=axes[0], label="month")

axes[1].scatter(sample_scatter["x1"], sample_scatter["x2"],
                alpha=0.3, s=5, color="purple")
axes[1].set_title("x1 (compressor freq) vs x2")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")

plt.tight_layout()
plt.show()

## 6. Device & Geographic Analysis

In [ ]:
dev_stats = (
    train.groupby("deviceId")
    .agg(mean_x2=("x2", "mean"), lat=("latitude", "first"), lon=("longitude", "first"))
    .dropna()
)

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(dev_stats["lon"], dev_stats["lat"],
                c=dev_stats["mean_x2"], cmap="plasma", s=30, alpha=0.8)
plt.colorbar(sc, ax=ax, label="mean x2")
ax.set_title("Device locations coloured by mean x2")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
N_CLUSTERS = 6
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
coords = devices[["latitude", "longitude"]].fillna(devices[["latitude","longitude"]].mean())
devices["geo_cluster"] = km.fit_predict(coords)

# Propagate to df, avoiding duplicate column issues
if "geo_cluster" in df.columns:
    df = df.drop(columns=["geo_cluster"])
df = df.merge(devices[["deviceId", "geo_cluster"]], on="deviceId", how="left")

# Refresh train slice
train = df[df["period"] == "train"].copy()
train["month"] = train["timedate"].dt.month
train["hour"]  = train["timedate"].dt.hour
train["day_of_week"] = train["timedate"].dt.dayofweek
train["month_label"] = train["timedate"].dt.to_period("M").astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cl, grp in devices.groupby("geo_cluster"):
    axes[0].scatter(grp["longitude"], grp["latitude"], label=f"Cluster {cl}", s=20)
axes[0].set_title(f"KMeans geographic clusters (k={N_CLUSTERS})")
axes[0].legend(fontsize=7)

sns.boxplot(data=train, x="geo_cluster", y="x2", ax=axes[1], showfliers=False)
axes[1].set_title("x2 by geo_cluster")
plt.tight_layout()
plt.show()

## 7. Data Engineering

In [ ]:
df = df.sort_values(["deviceId", "timedate"]).reset_index(drop=True)

# Temporal features
df["month"]       = df["timedate"].dt.month
df["day_of_week"] = df["timedate"].dt.dayofweek
df["hour"]        = df["timedate"].dt.hour
df["day_of_year"] = df["timedate"].dt.dayofyear
df["is_weekend"]  = df["day_of_week"] >= 5

def month_to_season(m):
    if m in (12, 1, 2): return 1  # winter
    if m in (3, 4, 5):  return 2  # spring
    if m in (6, 7, 8):  return 3  # summer
    return 4                       # autumn

df["season"] = df["month"].map(month_to_season)
print("Temporal features added.")

In [ ]:
# Lag & rolling features (per device)
grp = df.groupby("deviceId", sort=False)["x2"]

df["x2_lag_1"]   = grp.shift(1)
df["x2_lag_12"]  = grp.shift(12)
df["x2_lag_288"] = grp.shift(288)

df["x2_roll_mean_12"]  = grp.transform(lambda s: s.shift(1).rolling(12,  min_periods=1).mean())
df["x2_roll_std_12"]   = grp.transform(lambda s: s.shift(1).rolling(12,  min_periods=2).std())
df["x2_roll_mean_288"] = grp.transform(lambda s: s.shift(1).rolling(288, min_periods=1).mean())

df["t1_roll_mean_12"] = (
    df.groupby("deviceId", sort=False)["t1"]
    .transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())
)

print("Lag & rolling features added.")

In [ ]:
# Device stats from train only (no leakage)
train_mask = df["period"] == "train"
device_stats = (
    df[train_mask]
    .groupby("deviceId")["x2"]
    .agg(device_mean_x2="mean", device_std_x2="std")
    .reset_index()
)

# Avoid duplicate columns on re-run
for col in ["device_mean_x2", "device_std_x2"]:
    if col in df.columns:
        df = df.drop(columns=[col])

df = df.merge(device_stats, on="deviceId", how="left")
print(f"device_mean_x2 nulls: {df['device_mean_x2'].isna().sum()}")

In [ ]:
# Monthly aggregation — the actual prediction target
df["year_month"] = df["timedate"].dt.to_period("M")

monthly_target = (
    df[train_mask]
    .groupby(["deviceId", "year_month"])["x2"]
    .mean()
    .reset_index()
    .rename(columns={"x2": "mean_x2_per_month"})
)
print("Monthly target aggregation (train):")
monthly_target.head(5)

## 8. Feature Importance Preview

In [ ]:
FEATURE_COLS = [
    "month", "day_of_week", "hour", "day_of_year", "is_weekend", "season",
    "x1", "x3",
    "t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t10", "t11", "t12", "t13",
    "x2_lag_1", "x2_lag_12", "x2_lag_288",
    "x2_roll_mean_12", "x2_roll_std_12", "x2_roll_mean_288",
    "t1_roll_mean_12",
    "device_mean_x2", "device_std_x2",
    "geo_cluster", "latitude", "longitude",
    "deviceType",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

train_fi = df[train_mask].copy()

# Encode categoricals
for col in ["x3", "deviceType", "geo_cluster"]:
    if col in train_fi.columns:
        le = LabelEncoder()
        train_fi[col] = le.fit_transform(train_fi[col].astype(str))

X = train_fi[FEATURE_COLS].fillna(-999)
y = train_fi["x2"].fillna(0)

N = min(200_000, len(X))
idx = np.random.choice(len(X), N, replace=False)
X_s, y_s = X.iloc[idx], y.iloc[idx]

print(f"Fitting RandomForest on {N:,} rows, {len(FEATURE_COLS)} features...")

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_s, y_s)

importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
importances.head(20).plot.bar(ax=ax, color="steelblue")
ax.set_title("Top-20 feature importances (RandomForest)")
ax.set_ylabel("Importance")
plt.tight_layout()
plt.show()

print(importances.head(20))

## 9. Save Engineered Dataset

In [ ]:
# Convert Period dtype to string for parquet compatibility
df["year_month"] = df["year_month"].astype(str)

OUT_PATH = "data/eda_sample.parquet"
df.to_parquet(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Shape: {df.shape}")
print("\nFinal columns:")
print(df.columns.tolist())